# Export HILDA+ Land Cover for Lithuania

Exports:
- **PNG rasters** (dashboard style)
- **GeoTIFF rasters** (`rasters/hilda/geotiff/`) for web display – zoomable
- **CSV** per-class area counts

Run all cells. Use a kernel where rasterio works (e.g. Jupyter in landcover2) for GeoTIFF export.

In [1]:
from pathlib import Path
import json
from shapely.geometry import shape, Point
from shapely.ops import unary_union
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.transform import from_bounds
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

BASE = Path(r"C:\Users\matas\Desktop\LEI\Data")
LT_GEOJSON = BASE / "lt_boundary_admin.json"
HILDA_NC = BASE / "Hilda" / "Version 2.0" / "Winkler-etal_2025_allfiles" / "hildap_vGLOB-2.0_netCDF_extended-time" / "hildaplus_GLOB-2-0_states.nc"
OUT_RASTERS = BASE / "rasters" / "hilda"
OUT_GEOTIFF = OUT_RASTERS / "geotiff"
OUT_CSV = BASE / "outputs" / "hilda_lithuania_timeseries.csv"
OUT_RASTERS.mkdir(parents=True, exist_ok=True)
OUT_GEOTIFF.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with open(LT_GEOJSON, "r", encoding="utf-8") as f:
    lt_data = json.load(f)
geoms = [shape(feat["geometry"]) for feat in lt_data["features"]]
lt_geom = unary_union(geoms)

def polygon_mask(lat_vals, lon_vals, geom):
    h, w = len(lat_vals), len(lon_vals)
    mask = np.zeros((h, w), dtype=bool)
    for i, lat in enumerate(lat_vals):
        for j, lon in enumerate(lon_vals):
            mask[i, j] = geom.contains(Point(lon, lat))
    return mask

LAT_MIN, LAT_MAX = 53.5, 56.6
LON_MIN, LON_MAX = 20.5, 26.7

groups = {"Water": [0, 77], "Urban": [11], "Agriculture": [22, 23, 24, 33], "Forest": [40, 41, 42, 43, 44, 45, 55]}
five_map = {}
for c in groups["Water"]: five_map[c] = 1
for c in groups["Urban"]: five_map[c] = 3
for c in groups["Agriculture"]: five_map[c] = 4
for c in groups["Forest"]: five_map[c] = 5

class_names = {1: "Water", 2: "Wetlands", 3: "Urban", 4: "Agriculture", 5: "Forest"}
colors = ["#4DA6FF", "#7B68EE", "#FF4D4D", "#FFD24D", "#228B22"]
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = matplotlib.colors.Normalize(vmin=1, vmax=5)

def reclass_to_five(layer):
    data = layer.values
    out = np.full_like(data, fill_value=np.nan, dtype="float32")
    for orig, unified in five_map.items():
        out[data == orig] = unified
    return out

print("Opening HILDA dataset:", HILDA_NC)
ds = xr.open_dataset(HILDA_NC, chunks="auto")
da = ds["LULC_states"]
lat_name, lon_name = "latitude", "longitude"
lats = da[lat_name].values
lat_slice = slice(LAT_MAX, LAT_MIN) if (len(lats) > 1 and lats[0] > lats[-1]) else slice(LAT_MIN, LAT_MAX)
da_lt = da.sel(**{lat_name: lat_slice, lon_name: slice(LON_MIN, LON_MAX)})
lat_vals = da_lt[lat_name].values
lon_vals = da_lt[lon_name].values
mask = polygon_mask(lat_vals, lon_vals, lt_geom)

time_vals = da_lt["time"].values
year_to_ti = {}
for ti, t_val in enumerate(time_vals):
    y = int(round(float(t_val)))
    if 1910 <= y <= 2020:
        year_to_ti[y] = ti
years = np.array(sorted(year_to_ti.keys()), dtype=int)
print("Exporting HILDA years:", years[:10], "...", years[-5:])

Opening HILDA dataset: C:\Users\matas\Desktop\LEI\Data\Hilda\Version 2.0\Winkler-etal_2025_allfiles\hildap_vGLOB-2.0_netCDF_extended-time\hildaplus_GLOB-2-0_states.nc
Exporting HILDA years: [1910 1911 1912 1913 1914 1915 1916 1917 1918 1919] ... [2016 2017 2018 2019 2020]


In [2]:
records = []
for year in years:
    layer = da_lt.isel(time=year_to_ti[int(year)])
    arr = reclass_to_five(layer)
    arr_masked = np.where(mask, arr, np.nan)

    flat = arr_masked[np.isfinite(arr_masked)].astype(int)
    if flat.size:
        uniq, cnts = np.unique(flat, return_counts=True)
        for cls_id, cnt in zip(uniq, cnts):
            records.append((int(year), int(cls_id), class_names.get(int(cls_id), f"class_{cls_id}"), int(cnt)))

    rgba = cmap(norm(arr_masked))
    out_png = OUT_RASTERS / f"hilda_{int(year)}.png"
    plt.imsave(out_png, rgba)

    h, w = arr_masked.shape
    west, east = float(np.min(lon_vals)), float(np.max(lon_vals))
    south, north = float(np.min(lat_vals)), float(np.max(lat_vals))
    arr_uint8 = np.where(np.isfinite(arr_masked), arr_masked.astype(np.uint8), 0)
    transform = from_bounds(west, south, east, north, w, h)
    out_tif = OUT_GEOTIFF / f"hilda_{int(year)}.tif"
    with rasterio.open(out_tif, "w", driver="GTiff", height=h, width=w, count=1,
                      dtype=arr_uint8.dtype, crs="EPSG:4326", transform=transform, nodata=0) as dst:
        dst.write(arr_uint8, 1)

    if int(year) % 20 == 0:
        print(f"  Saved {out_png}, {out_tif}")

print("Done.")

C:\Users\matas\AppData\Local\Temp\ipykernel_13660\2133175291.py:21: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_masked), arr_masked.astype(np.uint8), 0)


  Saved C:\Users\matas\Desktop\LEI\Data\rasters\hilda\hilda_1920.png, C:\Users\matas\Desktop\LEI\Data\rasters\hilda\geotiff\hilda_1920.tif
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\hilda\hilda_1940.png, C:\Users\matas\Desktop\LEI\Data\rasters\hilda\geotiff\hilda_1940.tif
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\hilda\hilda_1960.png, C:\Users\matas\Desktop\LEI\Data\rasters\hilda\geotiff\hilda_1960.tif
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\hilda\hilda_1980.png, C:\Users\matas\Desktop\LEI\Data\rasters\hilda\geotiff\hilda_1980.tif
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\hilda\hilda_2000.png, C:\Users\matas\Desktop\LEI\Data\rasters\hilda\geotiff\hilda_2000.tif
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\hilda\hilda_2020.png, C:\Users\matas\Desktop\LEI\Data\rasters\hilda\geotiff\hilda_2020.tif
Done.


In [3]:
df = pd.DataFrame(records, columns=["year", "class_id", "class_name", "count"])
df.to_csv(OUT_CSV, index=False)
print("Saved CSV:", OUT_CSV)
df.head(10)

Saved CSV: C:\Users\matas\Desktop\LEI\Data\outputs\hilda_lithuania_timeseries.csv


,year,class_id,class_name,count
0,1910,1,Water,36
1,1910,3,Urban,351
2,1910,4,Agriculture,83825
3,1910,5,Forest,5617
4,1911,1,Water,36
5,1911,3,Urban,352
6,1911,4,Agriculture,83886
7,1911,5,Forest,5555
8,1912,1,Water,36
9,1912,3,Urban,351
